In [3]:
import subprocess
def gpu_available():
    try:
        subprocess.check_output(['nvidia-smi'])
        return True
    except Exception:
        return False
    
import platform
def is_linux():
    try:
        return platform.system() == 'Linux'
    except Exception:
        return False

In [4]:
gpu=gpu_available()
#gpu=False
linux=is_linux()
if gpu is True and linux is True:
    print('Linux detected')
    print('GPU detected')
    import pandas as pd
    import cudf.pandas
    cudf.pandas.install()
    import cuml
    from cuml.naive_bayes import CategoricalNB

else: 
    print(f"GPU: {gpu}\nLinux: {linux}")
    print(f"CuDF not available. Running Pandas")
    import pandas as pd
    from sklearn.naive_bayes import CategoricalNB


import joblib
from joblib import dump, load, Parallel, delayed
import scipy.stats
from scipy.stats import chi2_contingency
from scipy.stats import chisquare
from scipy.special import chdtrc
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

GPU: True
Linux: False
CuDF not available. Running Pandas


In [5]:
behavior='shopping_behavior_updated.csv'
df=pd.read_csv(behavior)
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [6]:
mymap={'Yes':1,'No':0}
for col in ['Subscription Status', 'Discount Applied']:
    df[col]=df[col].map(mymap).astype('object')

In [ ]:
#this implements an sklearn algorythm under the hood. there is a custom one meant to be more easily adapted to high level processors
class NBClassifier:
    def __init__(self):
        self.model=None
        self.OrdinalEncoded=False
        self.col_keys=None
        self.y_keys=None
        
    def fit(self,X,y):
        """

        """
        #print(f"X dtyp: {type(X)}, y dtype: {y.dtype}")
        X_=pd.DataFrame(X.copy())
        y_=pd.Series(y.copy())
        #print(f"X dtyp: {type(X_)}, y dtype: {y_.dtype}")
        self.OrdinalEncoded=True
        self.y_keys=pd.Series(y_.unique()).to_frame().reset_index(drop=False)
        self.y_keys.columns=['int_keys','cat_values']
        self.y_keys=self.y_keys.set_index('cat_values')
        y_.name='y' if not y_.name else y_.name
        y_=y_.to_frame().reset_index(drop=False).set_index(y_.name)
        y_.columns=['temp']
        y_=y_.join(self.y_keys)
        y_=y_.drop(columns='temp')
        self.y_keys=self.y_keys.reset_index(drop=False).set_index('int_keys') # so it will already be indexed for reasignment
        self.col_keys={}
        for col in X_.columns:
            self.col_keys[col]=pd.Series(X_[col].unique()).to_frame().reset_index(drop=False)
            self.col_keys[col].columns=['int_keys','cat_values']
            self.col_keys[col]=self.col_keys[col].set_index('cat_values')
            vector=X_[col].copy().to_frame().reset_index(drop=False)
            vector=vector.set_index(col)
            vector=vector.join(self.col_keys[col])
            X_[col]=vector['int_keys'].values
            del vector
            self.col_keys[col]=self.col_keys[col].reset_index(drop=False).set_index('int_keys')  # so it's ready for reasignment"""
        gpu=False
        if linux and gpu: 
            params={'alpha':1.0,'fit_prior':True,'output_type':'cudf'}
            self.model=CategoricalNB(**params)
        else: 
            params={'alpha':1.0,'fit_prior':True}
            self.model=CategoricalNB(**params)##########===========================FETCH MODEL
            #######################################################################################################
            ########################################################################################################
        self.model.fit(X_,y_)

    def predict(self,X):
        """

        """
        X_=pd.DataFrame(X)
        if self.col_keys is not None:
            for col in X_.columns:
                vector=X_[col].copy().to_frame()#.reset_index(drop=False)
                #vector=vector.set_index(col)
                #vector.columns=['temp']
                print(vector)
                print('........')
                print(self.col_keys[col])
                vector=vector.join(self.col_keys[col])
                print(vector)
                vector=vector.drop(columns='temp')
                X_[col]=vector.values
                del vector
        y_pred_keys = pd.Series(self.model.predict(X_)).to_frame().reset_index(drop=False)
        y_pred_keys.columns=['temp','pred']
        y_pred_keys = y_pred_keys.set_index('pred')
        y_pred_keys = y_pred_keys.join(self.y_col_keys)
        y_pred_keys = y_pred_keys.drop(columns='temp')
        return y_pred_keys.values
    
    def predict_proba(self,X):
        """

        """
        X_=pd.DataFrame(X.copy())
        return self.model.predict_proba(X_)
        
      

In [8]:
nbc=NBClassifier()
X=df[['Gender']]
y=df['Item Purchased']
nbc.fit(X,y)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [9]:
nbc.predict(X)

      Gender
0       Male
1       Male
2       Male
3       Male
4       Male
...      ...
3895  Female
3896  Female
3897  Female
3898  Female
3899  Female

[3900 rows x 1 columns]
........
         cat_values
int_keys           
0              Male
1            Female
      Gender cat_values
0       Male       Male
1       Male     Female
2       Male        NaN
3       Male        NaN
4       Male        NaN
...      ...        ...
3895  Female        NaN
3896  Female        NaN
3897  Female        NaN
3898  Female        NaN
3899  Female        NaN

[3900 rows x 2 columns]


KeyError: "['temp'] not found in axis"

In [ ]:
class NBayesClassifier:
    def __init__(self,method:str='bayes_probas'):
        self.Prob_y_DataFrame=None
        self.Prob_X_given_y_dict=None
        self.method=method       # can be of 'bayes_probas': complete bayes theorem, or 'log_classifier': classification only
        self.Py_cols=None
        self.result_key=None


    def build_key(self,X):
        
        X_=X.copy()
        X_=X_.astype('category')
        """
                    P(X|y)*P(y)
        ------------------------------------
        P(X|not y)*P(not y)  +   P(X|y)*P(y)
        """        
        result_key=X_.groupby(list(X_.columns),as_index=False,observed=True).all()   #OBSERVED TRUE BECAUSE THIS WILL BE DONE AT PREDICT, NOT FIT
        ## HEADS OF P(Y) COLUMNS THAT WILL BE UPDATED IN THE FOLLOWING LOOP 
        Py_cols=list(self.Prob_y_DataFrame.index)     # 
        self.Py_cols=Py_cols
        ## NAMES OF INTERMEDIATE COLUMNS
        intermediate_y_cols=[f"{col}_intermediate" for col in Py_cols]
        ## THE PY COLS THAT WILL REMAIN
        result_key[Py_cols]=np.nan 

        
        # BEGIN LOOP
        for col in X_.columns:
            #print(col)
            this_cols_PXy_key=self.Prob_X_given_y_dict[col]
            ####
            ## MAKE COL A TEMPORARY INDEX IN RESULT_KEY FOR THIS JOIN OPERATION
            result_key=result_key.set_index(col) 
            this_cols_PXy_key=this_cols_PXy_key.T
            self.dfd,self.fdf=result_key,this_cols_PXy_key
            #
            #print(f"       type(result_key.index): {type(result_key.index)}")
            #print(f"type(this_cols_PXy_key.index): {type(this_cols_PXy_key.index)}")
            #display(result_key)
            """            
            print(f"type(this_cols_PXy_key): {type(this_cols_PXy_key)}") 
            print(f"this_cols_PXy_key.index: {this_cols_PXy_key.index}")"""
            #display(this_cols_PXy_key)
            print('hi')
            result_key=result_key.join(this_cols_PXy_key,how='left',rsuffix='_intermediate',validate='m:1') 
            print('bye')
            #print('at join\n\n',result_key)
            ## RESET INDEX BACK (so that the column is preserved )
            result_key=result_key.reset_index(drop=False)
            #print('at reutrned index\n\n',result_key)
            ####
            # UPDATE PY COLS USING THE INTERMEDIATE INFO
            result_key[Py_cols] *= result_key[intermediate_y_cols] #X wise products
            #print('at product\n\n',result_key)
            result_key[Py_cols] = result_key[Py_cols].fillna(pd.DataFrame(result_key[intermediate_y_cols].values))   #update nan with observations col's observations
            #print('at fillna\n\n',result_key)
            result_key          = result_key.drop(columns=intermediate_y_cols)      # drop intermediate columns 
            #print('at dropped intermediat\n\n',result_key)
        #### END LOOP
        # MULTIPLY BY PY
        result_key[Py_cols] = result_key[Py_cols].multiply(self.Prob_y_DataFrame['P(y)'],axis=1)
        #...........................................................................................................................
        # TRUE BAYES THEOREM
        if self.method == 'bayes_probas':
            ###====> CALCULATE P(X|not y) * P(not y) 
            # HEADS OF P(NOT Y) COLUMNS THAT WILL BE UPDATED IN THE FOLLOWING LOOP 
            P_not_y_cols=list(self.Prob_y_DataFrame.index)     #
            ## NAMES OF NOT_Y_INTERMEDIATE COLUMNS
            intermediate_not_y_cols=[f"{col}_not_y_intermediate" for col in P_not_y_cols]
            ## THE PY COLS THAT WILL REMAIN
            result_key[P_not_y_cols]=np.nan 
            # BEGIN LOOP
            for col in X_.columns:
                if col not in self.Prob_X_given_y_dict:
                    raise ValueError(f"Column '{col}' not seen during fit.")
                col_PXnot_y=self.Prob_X_given_y_dict[col]
                col_PXnot_y=1-col_PXnot_y
                col_PXnot_y=col_PXnot_y.T
                ####
                ## MAKE COL A TEMPORARY INDEX IN RESULT_KEY FOR THIS JOIN OPERATION
                result_key=result_key.set_index(col) 
                result_key=result_key.join(col_PXnot_y,how='left',rsuffix='_not_y_intermediate')
                ## RESET INDEX BACK (so that the column is preserved )
                result_key=result_key.reset_index(drop=False)
                ####
                # UPDATE PY COLS USING THE INTERMEDIATE INFO
                result_key[P_not_y_cols] *= result_key[intermediate_not_y_cols]   #X wise products
                result_key[P_not_y_cols] = result_key[P_not_y_cols].fillna(result_key[intermediate_not_y_cols])   #update nan with observations col's observations
                result_key          = result_key.drop(columns=intermediate_not_y_cols)      # drop intermediate columns 
            #### END LOOP
            # MULTIPLY BY PY
            result_key[P_not_y_cols] = result_key[P_not_y_cols].multiply(self.Prob_y_DataFrame['P(not_y)'],axis=1)
            #............................................................................................

            result_key[Py_cols]=result_key[Py_cols] / ( result_key[Py_cols] + result_key[P_not_y_cols] )
            result_key = result_key.drop(columns=P_not_y_cols)        
            #............................................................................................
            ###
        ### FIND MAX P
        result_key['max_p_index'] = np.argmax(result_key[Py_cols],axis=1)
        # USE IT(max p) AS AN INDEX FOR JOIN
        result_key = result_key.set_index('max_p_index')
        # CREATE A PREDICTION KEY TO JOIN
        prediction_key=pd.DataFrame(Py_cols,index=[i for i in range(len(Py_cols))],columns=['prediction'])
        # PERFORM THE JOIN
        suffix = ''
        result_key = result_key.join(prediction_key,how='left',rsuffix=suffix)

        #REMOVE UNECESSARY COLUMNS
        if self.method=='log_classifier':
            result_key=result_key.drop(columns=Py_cols)

        self.result_key=result_key



    def fit(self,X,y):
        """ 

        """
        X_=X.copy()
        y_=y.copy()
       
        #convert y to category to ensure observed=False is used
        y_=y_.astype('category')
        y_.name= y_.name if y_.name is not None else 'y'
        py=y_.groupby(y_,observed=False).size().reset_index(drop=False,name='size').sort_values(by=y_.name)
        py['P(y)']=py['size']/py['size'].sum()
        py['P(not_y)']= 1 - py['P(y)']
        py=py[[y_.name,'P(not_y)','P(y)']]  
        py=py.set_index(y_.name)
        self.Prob_y_DataFrame=py 

        X_=X_.astype('category')
        X_given_y_data_dict={}
        for col in X_.columns:
            curr=X_[[col]]
            #print(curr.index,y_.index)
            curr=curr.join(y_)
            #curr=curr.groupby([col,y_.name],as_index=False,observed=False).size()  ###############< -----  looked obsolete...........................
            #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html
            #https://docs.rapids.ai/api/cudf/stable/user_guide/api_docs/api/cudf.dataframe.pivot/#cudf.DataFrame.pivot
            curr=curr.pivot_table(index=y_.name,columns=col,observed=False,aggfunc=['size']) #removed values='size'   #################<---------------because this may do that. however, i need to experiment
            #curr=curr.reset_index(drop=True)
            #curr=curr.set_index(y.name)
            #print(curr)
            if self.method=='log_classifier':
                 curr=curr.fillna(0)
                 curr=curr+1
            curr=curr.div(curr.sum(axis=1),axis='index')
            X_given_y_data_dict[col]=curr
        self.Prob_X_given_y_dict=X_given_y_data_dict

 

    def predict(self,X):
        ###
        # edge case ==> raise ValueError() if X not seen at fit
        ###
        X_=X
        if self.result_key is None:
            print('building_key 1')
            self.build_key(X_)
        elif any( X_.values not in self.result_key[X_.columns].values, axis = 'index'):            
            print('building_key 2')
            self.build_key(X_)
            if any( X_.values not in self.result_key[X_.columns].values, axis = 'index'):
                raise ValueError(f"X contains data not seen at fit")
        pred_df=X_.merge(self.result_key,on=list(X_.columns),how='left')
        res=pred_df['prediction'].copy()
        del X_
        return res



In [ ]:
pcl=NBayesClassifier()
X=df[['Gender']]
y=df['Season']
pcl.fit(X,y)

RangeIndex(start=0, stop=3900, step=1) RangeIndex(start=0, stop=3900, step=1)


In [ ]:
pcl.predict(X)

building_key 1
hi


InvalidIndexError: slice(None, None, None)

In [ ]:
pcl.Prob_y_DataFrame


,P(not_y),P(y)
Season,,
Fall,0.750000,0.250000
Spring,0.743846,0.256154
Summer,0.755128,0.244872
Winter,0.751026,0.248974


In [ ]:
pcl.Prob_X_given_y_dict


{'Gender':             size          
 Gender    Female      Male
 Season                    
 Fall    0.336410  0.663590
 Spring  0.316316  0.683684
 Summer  0.312042  0.687958
 Winter  0.315139  0.684861}

In [ ]:
pcl.method


'bayes_probas'

In [ ]:
pcl.Py_cols


['Fall', 'Spring', 'Summer', 'Winter']

In [ ]:
pcl.result_key

# ============================================================================================
expected values for y given X  
# ============================================================================================  
pareto is worth considering to model 'wealth'  